# Circuit Visualization — Differential Pair

This notebook demonstrates analog circuit schematic generation with **Schemdraw** and interactive MOSFET I-V curve sweeps with **Plotly**.

## Concepts
- Differential pair: two matched transistors sharing a tail current source
- MOSFET I-V characteristics: $I_{DS}$ vs $V_{DS}$ for varying $V_{GS}$
- Transistor sizing effect on drive strength

In [ ]:
import schemdraw
import schemdraw.elements as elm
from IPython.display import SVG, display

# Draw a differential pair schematic
with schemdraw.Drawing(file='diffpair.svg') as d:
    d.config(unit=0.5)
    elm.Ground().at((0,0))
    elm.SourceI().up().at((0,0)).label('$I_{tail}$')
    elm.Line().right().at((0,2)).length(1)
    elm.Line().up().at((-1,2)).length(1)
    elm.Line().up().at((1,2)).length(1)
    Q1 = elm.BjtNpn(circle=True).right().at((-1,3)).label('$Q_1$')
    Q2 = elm.BjtNpn(circle=True).right().at((1,3)).label('$Q_2$')
    elm.Line().up().at((-1,4.5)).length(1)
    elm.Line().up().at((1,4.5)).length(1)
    elm.Resistor().right().at((-1,5.5)).label('$R_D$')
    elm.Resistor().right().at((1,5.5)).label('$R_D$')
    elm.Line().right().at((0,2)).length(2)
    elm.Ground().at((2,0))

display(SVG('diffpair.svg'))

## Interactive MOSFET I-V Sweep

Use the slider below to vary transistor width $W$ and observe how the saturation current scales.

In [ ]:
import plotly.express as px
import numpy as np
import pandas as pd
from ipywidgets import interact, FloatSlider

def mosfet_iv(W_um=5.0, L_um=0.5, Vgs_steps=5):
    """Generate MOSFET I-V curves for given W/L."""
    Vds = np.linspace(0, 3, 200)
    Vth = 0.4
    muCox = 200e-6  # A/V^2
    data = []
    for i, Vgs in enumerate(np.linspace(0.6, 1.8, Vgs_steps)):
        Ids = np.where(
            Vds < Vgs - Vth,
            muCox * (W_um/L_um) * ((Vgs - Vth) * Vds - 0.5 * Vds**2),
            0.5 * muCox * (W_um/L_um) * (Vgs - Vth)**2
        )
        for v, i_val in zip(Vds, Ids):
            data.append({'Vds (V)': v, 'Ids (A)': i_val, 'Vgs': f'{Vgs:.2f}V'})
    df = pd.DataFrame(data)
    fig = px.line(
        df, x='Vds (V)', y='Ids (A)', color='Vgs',
        title=f'MOSFET I-V (W={W_um}µm, L={L_um}µm)',
        template='plotly_white'
    )
    fig.update_layout(hovermode='x unified')
    fig.show()

interact(
    mosfet_iv,
    W_um=FloatSlider(min=1, max=20, step=1, value=5, description='W (µm)'),
    L_um=FloatSlider(min=0.18, max=2, step=0.18, value=0.5, description='L (µm)'),
    Vgs_steps=FloatSlider(min=3, max=10, step=1, value=5, description='Vgs steps')
)

## Expected Output

- Above: SVG schematic of a differential pair
- Below: Interactive Plotly figure with I-V curves that update when W/L sliders change